In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [5]:
import zipfile

zip_path = "/content/drive/MyDrive/PFM_AquaVision/archive (1).zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully")

Dataset extracted successfully


In [7]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    print(root)

/content/dataset
/content/dataset/Micro_Organism
/content/dataset/Micro_Organism/Yeast
/content/dataset/Micro_Organism/Euglena
/content/dataset/Micro_Organism/Spiral_bacteria
/content/dataset/Micro_Organism/Amoeba
/content/dataset/Micro_Organism/Spherical_bacteria
/content/dataset/Micro_Organism/Paramecium
/content/dataset/Micro_Organism/Rod_bacteria
/content/dataset/Micro_Organism/Hydra


In [13]:
# Dataset path
DATASET_PATH = "/content/dataset/Micro_Organism"

# Data Generator
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# MobileNetV2 Pretrained
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(8, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

model.save("microbe_model.h5")

print("Model saved successfully!")

Found 633 images belonging to 8 classes.
Found 156 images belonging to 8 classes.
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 58s 3s/step - accuracy: 0.3033 - loss: 1.9874 - val_accuracy: 0.5192 - val_loss: 1.4319
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.5529 - loss: 1.3474 - val_accuracy: 0.5833 - val_loss: 1.2214
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.6272 - loss: 1.1231 - val_accuracy: 0.6090 - val_loss: 1.1090
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.6682 - loss: 0.9780 - val_accuracy: 0.6090 - val_loss: 1.1108
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.6983 - loss: 0.8877 - val_accuracy: 0.6603 - val_loss: 1.1030


Model saved successfully!


In [14]:
from google.colab import files
files.download("microbe_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>